# 1. Κατεβάζουμε τα δεδομένα

In [ ]:
import json
import os
from gdown import download
import torch
def download_data(id, filename):
    if not os.path.exists(filename):
        download(id=id, output=filename, quiet=False)
    with open(filename, 'r', encoding='utf-8') as f:
        return json.load(f)
movies = download_data('1bwVnwW9FL4-pMUpOTzXOomHc150NncmW', 'movies.json')
levels = download_data('1O_vJ7aoWfakDKFCZb0zc51BjzUoLJrCW', 'levels.json')
embeddings = torch.tensor( download_data('1E1lHQQ09yQWSw-YWmsnRt8ukbV7-Gznr', 'embeddings.json') )

Downloading...
From: https://drive.google.com/uc?id=1bwVnwW9FL4-pMUpOTzXOomHc150NncmW
To: /content/movies.json
100%|██████████| 753k/753k [00:00<00:00, 28.9MB/s]
Downloading...
From: https://drive.google.com/uc?id=1O_vJ7aoWfakDKFCZb0zc51BjzUoLJrCW
To: /content/levels.json
100%|██████████| 3.81k/3.81k [00:00<00:00, 8.84MB/s]
Downloading...
From: https://drive.google.com/uc?id=1E1lHQQ09yQWSw-YWmsnRt8ukbV7-Gznr
To: /content/embeddings.json
100%|██████████| 4.97M/4.97M [00:00<00:00, 56.1MB/s]


# 2. Αρχικοποιούμε το μοντέλο.

Η σελίδα χρησιμοποιεί μια διακριτοποιημένη έκδοση οπότε μπορεί να έχει μικρές διαφορές με την python.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-small")
def embed_query(query):
    return model.encode([query], convert_to_numpy=False, convert_to_tensor=True)[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Διαφορετικά μπορείτε να χρησιμοποιήσετε αυτό το μοντέλο που είναι πιο κοντά στην υλοποίηση της σελίδας

In [ ]:
%pip install onnxruntime
import onnxruntime as ort
from transformers import AutoTokenizer
import transformers
import torch
import requests

url = "https://huggingface.co/Xenova/multilingual-e5-small/resolve/main/onnx/model_quantized.onnx"
resp = requests.get(url)
resp.raise_for_status()

with open("model_quantized.onnx", "wb") as f:
    f.write(resp.content)
tokenizer = AutoTokenizer.from_pretrained("Xenova/multilingual-e5-small")
session = ort.InferenceSession("model_quantized.onnx")

def embed_query(q):
    inp = tokenizer([q], return_token_type_ids=True)
    outputs = session.run(["last_hidden_state"], dict(inp))
    emb = torch.tensor( outputs[0] ).float().mean(1)
    emb /= emb.norm(dim=-1)
    return emb[0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.2 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

# 3. Λύνουμε το αντίστοιχο επίπεδο

In [ ]:
answers = {}

In [ ]:
levelid = 'level1' # 'level2' / 'level3' / 'level4'
level = levels[levelid]
target_order = torch.tensor([0,1,2,3,4])
E = embeddings[ level ] # embeddings ολων των ταινιών του επιπέδου
E.shape

torch.Size([15, 384])

In [ ]:
# Υπολογίστε το καλύτερο embedding

# Είτε μέσω κάποιου ερωτήματος
query = "θρίλερ, έγκλημα, μυστήριο, δράμα, αϋπνία, ενοχή, συνείδηση, συγκάλυψη, δολοφονία, Αλάσκα"
emb = embed_query(query)
emb.shape

# Είτε απευθείας
emb = torch.randn(384)
emb.shape

torch.Size([384])

In [ ]:
# Ομοιότητα και Διάταξη ταινιών (φθίνουσα διάταξη με similarity)
similarity = torch.cosine_similarity(E, emb)
ranking = similarity.argsort(descending=True)
similarity[ranking], ranking

(tensor([ 0.0375,  0.0173,  0.0054,  0.0024,  0.0015,  0.0014, -0.0070, -0.0164,
         -0.0169, -0.0238, -0.0319, -0.0363, -0.0369, -0.0400, -0.0555]),
 tensor([ 2,  3, 12, 14,  4, 13,  8,  6,  9,  7,  1, 10,  0, 11,  5]))

In [ ]:
# TODO: Βρείτε ένα query για κάθε επίπεδο ώστε οι πρώτες 5 ταινίες να είναι οι 0,1,2,3,4...
torch.all( ranking[:5] == target_order ).item()

False

In [ ]:
# ...και οι διαδοχικές διαφορές να είναι όσο το δυνατόν μεγαλύτερες
diffs = -similarity[ranking].diff()[:5]
print('Διαδοχικές Διαφορές {:.4f}, {:.4f}, {:.4f}, {:.4f}, {:.4f}'.format(*diffs) )
print('Ελάχιστη Διαφορά {:.4f}'.format(diffs.min()))

Διαδοχικές Διαφορές 0.0202, 0.0119, 0.0030, 0.0009, 0.0001
Ελάχιστη Διαφορά 0.0001


In [ ]:
answers[levelid] = {'embedding': emb.tolist()}

# 4. Αποθηκεύουμε τις απαντήσεις για υποβολή στο site

In [ ]:
import json
with open('answers.json', 'w') as f:
    json.dump(answers, f)

In [ ]:
from google.colab import files
files.download('answers.json')